In [80]:
##### Cleans FAO livestock production (primary products) data
# splits and sums into animal groupings

import os
import pandas as pd
from functools import reduce

In [73]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

# Import data
production = pd.read_csv(f"{cd}/Data/Raw/FAO_production/livestock_primary_production.csv")

animal_match = pd.read_csv(f"{cd}/Data/Correspondence_tables/livestock_products.csv")

country_codes = pd.read_csv(f"{cd}/Data/Correspondence_tables/country_names.csv", encoding="cp1252")

# Set save path
save_path = f"{cd}/Data/Clean/Production/FAO_production_by_animal.csv"


In [74]:
### Combine with animal match and split into seperate dfs

production = production.merge(animal_match, left_on='Item', right_on='FAO_Item', how='left')
production = production.merge(country_codes, left_on='Area Code (M49)', right_on='FAO_code', how='left')

# drop if missing ISO code
production = production.dropna(subset=['ISO3'])

# split by livestock layer
unique_layers = production['livestock_layer'].unique()

# Create a dictionary where keys are layer names and values are the corresponding DataFrames
prod_by_animal = {layer: production[production['livestock_layer'] == layer].copy() for layer in unique_layers}

cattle = prod_by_animal['cattle']
goats = prod_by_animal['goats']
sheep = prod_by_animal['sheep']
chicken = prod_by_animal['chicken']
pigs = prod_by_animal['pigs']
horses = prod_by_animal['horses']
ducks = prod_by_animal['ducks']
buffalo = prod_by_animal['buffalo']
other = prod_by_animal['all']

In [77]:
### Clean chicken data 
# some duplicates in eggs reported in 1000No and t 

# seperate out hen eggs, eggs, and other chicken
chicken_hen_eggs = chicken[chicken['Item'] == 'Hen eggs in shell, fresh'].copy()
chicken_other_eggs = chicken[chicken['Item'] == 'Eggs from other birds in shell, fresh, n.e.c.'].copy()

chicken_all_else = chicken[~chicken['Item'].isin(['Eggs from other birds in shell, fresh, n.e.c.','Hen eggs in shell, fresh'])].copy()

# split out # and tones
chicken_hen_eggs_num = chicken_hen_eggs[chicken_hen_eggs['Unit'] == '1000 No']
chicken_hen_eggs_t = chicken_hen_eggs[chicken_hen_eggs['Unit'] == 't']

chicken_other_eggs_num = chicken_other_eggs[chicken_other_eggs['Unit'] == '1000 No']
chicken_other_eggs_t = chicken_other_eggs[chicken_other_eggs['Unit'] == 't']

chicken_hen_eggs_num = chicken_hen_eggs_num.dropna(subset=['Value'])
chicken_hen_eggs_t   = chicken_hen_eggs_t.dropna(subset=['Value'])
chicken_other_eggs_num = chicken_other_eggs_num.dropna(subset=['Value'])
chicken_other_eggs_t   = chicken_other_eggs_t.dropna(subset=['Value'])

# isolate key columns
col_to_keep = ['ISO3', 'Value']
chicken_hen_eggs_num = chicken_hen_eggs_num[col_to_keep]
chicken_hen_eggs_t   = chicken_hen_eggs_t[col_to_keep]
chicken_other_eggs_num = chicken_other_eggs_num[col_to_keep]
chicken_other_eggs_t   = chicken_other_eggs_t[col_to_keep]

# convert eggs to t
chicken_hen_eggs_num['Value_t_from_n'] = chicken_hen_eggs_num['Value'] * 0.05
chicken_other_eggs_num['Value_t_from_n'] = chicken_other_eggs_num['Value'] * 0.05

# merge back together 
chicken_hen_eggs = chicken_hen_eggs_t.merge(chicken_hen_eggs_num, on='ISO3', how='left')
chicken_other_eggs = chicken_other_eggs_t.merge(chicken_other_eggs_num, on='ISO3', how='left')

# create new column which is final value filled with t_from_n if t is missing
chicken_hen_eggs['Value'] = chicken_hen_eggs['Value_x'].fillna(chicken_hen_eggs['Value_t_from_n'])
chicken_hen_eggs['Item'] = 'Hen eggs in shell, fresh'
chicken_hen_eggs['Unit'] = 't'

chicken_other_eggs['Value'] = chicken_other_eggs['Value_x'].fillna(chicken_other_eggs['Value_t_from_n'])
chicken_other_eggs['Item'] = 'Eggs from other birds in shell, fresh, n.e.c.'
chicken_other_eggs['Unit'] = 't'

# select final cols
final_col = ['ISO3', 'Item', 'Unit', 'Value']

chicken_hen_eggs = chicken_hen_eggs[final_col]
chicken_other_eggs = chicken_other_eggs[final_col]

chicken_all_else = chicken_all_else[final_col]

# Concatinate back together 
chicken_fixed = pd.concat([chicken_all_else, chicken_hen_eggs, chicken_other_eggs], ignore_index=True)

# Drop na
chicken_fixed = chicken_fixed.dropna()

# Sum by ISO3 
chicken_sum = chicken.groupby('ISO3', as_index=False).agg(chicken_t=('Value', 'sum'))

In [79]:
##### Clean all other animal dfs

dfs = [cattle, goats, sheep, pigs, horses, ducks, buffalo, other]
names = ['cattle', 'goats', 'sheep', 'pigs', 'horses', 'ducks', 'buffalo', 'other']

sum_dfs = {}

for df, name in zip(dfs, names):
    summed = df.groupby('ISO3', as_index=False).agg(**{f'{name}_t': ('Value', 'sum')})
    
    sum_dfs[name] = summed

In [85]:
##### Merge all df's 

# sum all but chicken
all_sum = reduce(lambda left, right: pd.merge(left, right, on='ISO3', how='outer'), sum_dfs.values())

# add chicken
animals = all_sum.merge(chicken_sum, on='ISO3', how='outer')

# fill missing with 0 for no production 
animals = animals.fillna(0)

# add sum variable 
animals['total_t'] = (animals['cattle_t'] + animals['goats_t'] + animals['sheep_t'] + 
                      animals['pigs_t'] + animals['horses_t'] + animals['ducks_t'] + 
                      animals['buffalo_t'] + animals['other_t'] + animals['chicken_t']
)

In [86]:
### Save
animals.to_csv(save_path, index=False)